# Hangman wordlist explorer

Play with the curated master wordlist and the `wordlist.py` loader.
Run from anywhere in the project (repo root, `math/`, or `math/notebooks/`)
— the setup cell finds `wordlist.py` and imports it.

The list is `data/wordlist_master.txt` (`word<TAB>zipf`, frequency-sorted),
built by `math/build_wordlist.py`. `zipf` is wordfreq's frequency scale:
higher = more common (4.0 ≈ 10/million, 3.0 ≈ 1/million, 2.0 = the file floor).
The loader's default cutoff is `DEFAULT_ZIPF = 2.5`.

In [1]:
from pathlib import Path
import sys

def find_math_dir():
    """Find the directory containing wordlist.py (the math/ dir)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent, cwd.parent.parent, cwd / 'math', cwd.parent / 'math'):
        if (candidate / 'wordlist.py').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find wordlist.py. Open this notebook from the project root, '
        'math/, or math/notebooks/.'
    )

math_dir = find_math_dir()
if str(math_dir) not in sys.path:
    sys.path.insert(0, str(math_dir))

import wordlist
from wordlist import load, tier, Word, DEFAULT_ZIPF, TIERS

print(f'imported wordlist from {math_dir}')
print(f'DEFAULT_ZIPF = {DEFAULT_ZIPF}')

imported wordlist from /Users/patrickliscio/My Documents/Pycharm Projects/Ballpark-Figures/hangman/math
DEFAULT_ZIPF = 2.5


## The whole list at a glance

In [2]:
full = wordlist._raw()   # every stored word, frequency order, no cutoff
print(f'{len(full):,} words total, zipf {full[-1].zipf:.2f} .. {full[0].zipf:.2f}')
print(f'{len(load()):,} words at the default cutoff (zipf >= {DEFAULT_ZIPF})')
print()
print('most common 15:', ', '.join(w.text for w in full[:15]))
print('rarest 15:     ', ', '.join(w.text for w in full[-15:]))

47,839 words total, zipf 2.00 .. 7.73
34,941 words at the default cutoff (zipf >= 2.5)

most common 15: the, to, and, of, a, in, i, is, for, that, you, it, on, with, this
rarest 15:      uninviting, upcountry, vacuoles, vend, victuals, voile, voyageurs, wackos, wallah, whetted, whimsically, whoso, wittily, wretchedness, yurts


## Tune the cutoff

Change `CUTOFF` and re-run to see how many words survive and what the
boundary looks like. Words *just above* the cutoff are the hardest ones
still included — a good gut-check for whether the cutoff feels right.

In [3]:
CUTOFF = 2.5   # <-- try 3.0, 2.8, 2.2, 2.0 ...

words = load(min_zipf=CUTOFF)
print(f'zipf >= {CUTOFF}: {len(words):,} words\n')

# 30 words sitting right at the cutoff (the marginal, hardest-kept words)
near = [w for w in words if w.zipf < CUTOFF + 0.06]
print(f'{len(near)} words within 0.05 of the cutoff — sample:')
print(', '.join(w.text for w in near[:30]))

zipf >= 2.5: 34,941 words

1548 words within 0.05 of the cutoff — sample:
abducting, abides, aboriginals, acrimonious, adorning, adverb, anarchic, arras, attache, backtracking, bandy, barnacle, barometric, beguiling, bendy, berets, bevy, bidirectional, blanked, breastfed, bronzed, brunettes, bunched, bureaucracies, bushings, buzzards, calipers, certifies, chaparral, chivalrous


## Filter by length and frequency

`load()` takes `min_len`, `max_len`, `min_zipf`, `max_zipf`. Handy combo:
a `min_zipf`/`max_zipf` *band* isolates "known but not trivially common"
words — good for a harder round.

In [4]:
# e.g. 6-letter words in the 'known but not everyday' band
sel = load(min_len=6, max_len=6, min_zipf=2.5, max_zipf=3.5)
print(f'{len(sel):,} six-letter words, zipf 2.5-3.5')
print(', '.join(w.text for w in sel[:40]))

2,919 six-letter words, zipf 2.5-3.5
ankles, avoids, badges, bridal, buckle, bursts, chilly, chords, clowns, cradle, crappy, fellas, haters, hitter, legged, mailed, mantra, mapped, mating, outing, peaked, primer, router, shovel, splits, toxins, twists, attire, barley, bitten, blasts, braces, bumped, catchy, diaper, erased, hailed, hiatus, ledger, lodged


## Distributions

Text histograms (no plotting deps). How word length and frequency are spread
across the list at the current cutoff — useful for sanity-checking difficulty.

In [5]:
from collections import Counter

def bar(count, scale):
    return '█' * int(count / scale)

words = load(min_zipf=CUTOFF)

print(f'Word length distribution (zipf >= {CUTOFF}):')
by_len = Counter(len(w.text) for w in words)
scale = max(by_len.values()) / 40
for length in range(min(by_len), max(by_len) + 1):
    c = by_len.get(length, 0)
    print(f'  {length:>2} | {bar(c, scale):<40} {c:,}')

print(f'\nWords per 0.5-zipf band:')
by_band = Counter((int(w.zipf * 2) / 2) for w in words)
scale = max(by_band.values()) / 40
for band in sorted(by_band, reverse=True):
    c = by_band[band]
    print(f'  {band:>4.1f} | {bar(c, scale):<40} {c:,}')

Word length distribution (zipf >= 2.5):
   1 |                                          26
   2 | █                                        160
   3 | █████                                    762
   4 | ███████████████                          2,187
   5 | █████████████████████████                3,530
   6 | ██████████████████████████████████       4,850
   7 | ████████████████████████████████████████ 5,569
   8 | █████████████████████████████████████    5,289
   9 | ████████████████████████████████         4,513
  10 | ████████████████████████                 3,364
  11 | ███████████████                          2,148
  12 | █████████                                1,313
  13 | █████                                    710
  14 | ██                                       302
  15 | █                                        147
  16 |                                          47
  17 |                                          18
  18 |                                          3
  19 |     

In [6]:
length = 19
sel = load(min_len=length)
words = [x.text for x in sel]
print(len(words))
print(words)
l = [(let, len([w for w in words if let in w])) for let in 'abcdefghijklmnopqrstuvwxyz']
x = sorted(l, key=lambda x: -x[1])
for i in x:
    print(i)

3
['counterintelligence', 'internationalization', 'uncharacteristically']
('e', 3)
('i', 3)
('l', 3)
('n', 3)
('r', 3)
('t', 3)
('a', 2)
('c', 2)
('o', 2)
('u', 2)
('g', 1)
('h', 1)
('s', 1)
('y', 1)
('z', 1)
('b', 0)
('d', 0)
('f', 0)
('j', 0)
('k', 0)
('m', 0)
('p', 0)
('q', 0)
('v', 0)
('w', 0)
('x', 0)


## Letter position patterns

For a chosen word length, tabulate for each letter every subset of positions
it occupies across the list — e.g. length 3, `e`: `___`, `e__`, `_e_`, `__e`,
`ee_`, `e_e`, `_ee`. Columns: `count` (# words) and `share` (fraction of words)
are unweighted; `freq` and `prob` weight each word by how common it is.
Logic lives in `letter_patterns.py`.

In [10]:
import letter_patterns

LENGTH = 2   # <-- pick a word length

patterns = letter_patterns.pattern_frequencies(LENGTH)
nw = patterns.attrs['n_words']
print(f'{nw:,} words of length {LENGTH}, {len(patterns):,} (letter, pattern) rows')
patterns.sort_values("share", ascending=False).head(50)

160 words of length 2, 84 (letter, pattern) rows


,letter,pattern,n_positions,count,share,freq,prob
82,z,__,0,158,0.98750,0.161904,0.999932
31,j,__,0,158,0.98750,0.161878,0.999774
54,q,__,0,157,0.98125,0.161905,0.999941
69,v,__,0,154,0.96250,0.161799,0.999286
79,y,__,0,153,0.95625,0.153528,0.948206
72,w,__,0,153,0.95625,0.158420,0.978416
66,u,__,0,152,0.95000,0.158316,0.977777
75,x,__,0,152,0.95000,0.161818,0.999403
44,n,__,0,152,0.95000,0.129497,0.799785
10,d,__,0,149,0.93125,0.159449,0.984774


In [8]:
# every position-subset for one letter, most common first
LETTER = 'e'
patterns[patterns.letter == LETTER].reset_index(drop=True)

,letter,pattern,n_positions,count,share,freq,prob
0,e,__,0,146,0.9125,0.143930,0.888928
1,e,e_,1,8,0.0500,0.000246,0.001517
2,e,_e,1,6,0.0375,0.017739,0.109556


In [9]:
# one letter, dropping the 'absent' (___) row — showing all columns
# (count/share unweighted, freq/prob usage-weighted)
patterns[(patterns.letter == LETTER) & (patterns.n_positions > 0)].reset_index(drop=True)

,letter,pattern,n_positions,count,share,freq,prob
0,e,e_,1,8,0.0500,0.000246,0.001517
1,e,_e,1,6,0.0375,0.017739,0.109556
